In [1]:
using LowLevelFEM, LinearAlgebra

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 3 for different dependency version already loaded, 2 for different Julia build configuration, 4 for file size changed)
Precompiling packages...
   9.1 s  ✓ LowLevelFEM
  1 dependency successfully precompiled in 10 seconds. 133 already precompiled.


In [ ]:
structured_rect_mesh(lx=10)
#openPreProcessor()
mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u)
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U), Ω="body")
f = ∫(U ⋅ [100,-100], Ω="body")
bc = BoundaryCondition("left", ux=1, uy=1) # <====
per = MultiPointConstraint(master="left", slave="right", ux=false, uy=true)
u = solveField(Symmetric(K), f, support=[bc], mpc=[per])
showDoFResults(u, name="u", factor=1, visible=true)
openPostProcessor();

XOpenIM() failed
Fontconfig warning: using without calling FcInit()


In [ ]:
structured_rect_mesh(lx=10)
#openPreProcessor()
mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u)
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U), Ω="body")
Kc = ∫(U ⋅ 1e4 ⋅ U, Γ="left") # <====
f = ∫(U ⋅ [100,-100], Ω="body")
per = MultiPointConstraint(master="left", slave="right", ux=false, uy=true)
u = solveField(Symmetric(K + Kc), f, mpc=[per])
showDoFResults(u, name="u", factor=1, visible=true)
openPostProcessor();

In [ ]:
structured_rect_mesh(lx=10)
#openPreProcessor()
mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u)
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U), Ω="body")
Kc = ∫(U ⋅ 1e4 ⋅ U, Γ="left")
s = ScalarField(U, "body", (x,y,z)->x/10)
f = ∫(U ⋅ [100,-100s], Ω="body")
e1 = VectorField(U, "right", [1,0,0]) # <====
cs = CoordinateSystem(e1, handedness=:left) # <====
per = MultiPointConstraint(master="left", slave="right", ux=true, uy=true)
u = solveField(Symmetric(K + Kc), f, mpc=[per], coordSys=[cs])
showDoFResults(u, name="u", factor=1, visible=true)
openPostProcessor();

In [14]:
structured_rect_mesh(lx=10, rightRemote=true)
#openPreProcessor()
mat = Material("body")
rem = Material("rightremote")
U = Field([mat,rem], type=:VectorField, dim=2, fieldName=:u)
Φ = Field([mat,rem], type=:ScalarField, dim=2, fieldName=:φ)
Ku = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U), Ω="body")
Ku += 0*∫(U ⋅ 1e4 ⋅ U, Γ="left")
mpc_U = MultiPointConstraint(master="rightremote", slave="right", field=U)
mpc_Φ = MultiPointConstraint(master="rightremote", slave="right", field=Φ)
R = rigidRotationMap(mpc_U, mpc_Φ)
Kuφ = Ku * R
Kφ = R' * Kuφ
K = SystemMatrix([ Ku Kuφ; Kuφ' Kφ])
fu = ∫(U ⋅ [0,0], Γ="rightremote")
fφ = ∫(Φ ⋅ 1000, Γ="rightremote")
f = SystemVector([fu, fφ])
e1 = VectorField(U, "right", [1,0,0])
cs = CoordinateSystem(e1, handedness=:right)
per = MultiPointConstraint(master="left", slave="right", ux=true, uy=true, field=U)
u0, φ = solveField(Symmetric(K), f, mpc=[per, mpc_U, mpc_Φ], coordSys=[cs])
u = u0 + R * φ
showDoFResults(u, name="u", factor=1, visible=true)
openPostProcessor();